# EXHEART - transport recalibration (standalone)

Run once in Colab (GPU). Recomputes the two temporal-transport arms under **leakage-free
out-of-fold calibration** and compares against the published (leaky) calibration:

- **Naive transport**: full 21-feature 2015 model applied to 2020 with mismatched encoding
  (the deliberate misalignment that drives the apparent collapse).
- **Harmonised transport**: 11-feature model with semantic feature/encoding alignment.

For each it prints the 2015 baseline sensitivity and the 2020 transport sensitivity at pt=0.12,
leaky vs honest. Discrimination (AUC) is calibration-invariant and unchanged. Expect the drift
and the collapse to be robust; absolute sensitivities shift up ~1-2 points under honest
calibration. Saves `transport_leakagefree.json`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
REPO='/content/drive/MyDrive/EXHEART_Research/exheart-research'; SEED=42; PT=0.12
D15=f'{REPO}/data/brfss2015/heart_disease_health_indicators_BRFSS2015.csv'
D20=f'{REPO}/data/brfss2020/heart_2020_cleaned.csv'

def build_mlp(d):
    inp=keras.Input(shape=(d,)); x=layers.Dense(256,activation='relu')(inp); x=layers.BatchNormalization()(x); x=layers.Dropout(0.3)(x)
    x=layers.Dense(128,activation='relu')(x); x=layers.BatchNormalization()(x); x=layers.Dropout(0.3)(x)
    x=layers.Dense(64,activation='relu')(x); x=layers.Dropout(0.2)(x); out=layers.Dense(1,activation='sigmoid')(x)
    m=keras.Model(inp,out); m.compile(optimizer=keras.optimizers.Adam(1e-3),loss='binary_crossentropy',metrics=[keras.metrics.AUC(name='auc')]); return m
_es=lambda:keras.callbacks.EarlyStopping(monitor='val_auc',patience=5,restore_best_weights=True,mode='max')

def train_stack(Xtr_a, ytr_a, Xtr_sc, balanced_meta):
    """Return fitted base learners, scaler-transformed access, meta, and OOF stack probs."""
    spw=(ytr_a==0).sum()/(ytr_a==1).sum(); cw=compute_class_weight('balanced',classes=np.array([0,1]),y=ytr_a); kcw={0:cw[0],1:cw[1]}
    def trees(): return (XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.05,scale_pos_weight=spw,eval_metric='logloss',random_state=SEED,n_jobs=-1),
                         LGBMClassifier(n_estimators=300,max_depth=6,learning_rate=0.05,class_weight='balanced',random_state=SEED,n_jobs=-1,verbose=-1),
                         RandomForestClassifier(n_estimators=200,max_depth=10,class_weight='balanced',random_state=SEED,n_jobs=-1))
    xg,lg,rf=trees(); xg.fit(Xtr_a,ytr_a); lg.fit(Xtr_a,ytr_a); rf.fit(Xtr_a,ytr_a)
    tf.random.set_seed(SEED); mf=build_mlp(Xtr_sc.shape[1]); mf.fit(Xtr_sc,ytr_a,epochs=50,batch_size=512,validation_split=0.1,class_weight=kcw,callbacks=[_es()],verbose=0)
    skf=StratifiedKFold(5,shuffle=True,random_state=SEED); oof=np.zeros((len(Xtr_a),4))
    for k,(tr,va) in enumerate(skf.split(Xtr_a,ytr_a)):
        a,b,c=trees(); a.fit(Xtr_a[tr],ytr_a[tr]); b.fit(Xtr_a[tr],ytr_a[tr]); c.fit(Xtr_a[tr],ytr_a[tr])
        tf.random.set_seed(SEED+k); mm=build_mlp(Xtr_sc.shape[1]); mm.fit(Xtr_sc[tr],ytr_a[tr],epochs=50,batch_size=512,validation_split=0.1,class_weight=kcw,callbacks=[_es()],verbose=0)
        oof[va]=np.column_stack([a.predict_proba(Xtr_a[va])[:,1],b.predict_proba(Xtr_a[va])[:,1],c.predict_proba(Xtr_a[va])[:,1],mm.predict(Xtr_sc[va],verbose=0).ravel()])
        print(f'    OOF fold {k+1}/5')
    meta=LogisticRegression(class_weight=('balanced' if balanced_meta else None),max_iter=1000,random_state=(SEED if balanced_meta else None)).fit(oof,ytr_a)
    return dict(xg=xg,lg=lg,rf=rf,mf=mf,meta=meta,oof=oof)

def sens(y,p):
    pred=(p>=PT).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred).ravel(); return round(tp/(tp+fn),3)
print('Setup done. GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ================= NAIVE TRANSPORT (full 21-feature, mismatched encoding) =================
print('NAIVE transport: training full 21-feature 2015 stack...')
d15=pd.read_csv(D15); TGT='HeartDiseaseorAttack'; FEATS=[c for c in d15.columns if c!=TGT]
X15=d15[FEATS].copy(); y15=d15[TGT].astype(int)
Xtr,Xte,ytr,yte=train_test_split(X15,y15,test_size=0.2,random_state=SEED,stratify=y15)
Xtr_a,Xte_a,ytr_a,yte_a=Xtr.values,Xte.values,ytr.values,yte.values
sc=StandardScaler().fit(Xtr_a); Xtr_sc=sc.transform(Xtr_a)
S=train_stack(Xtr_a,ytr_a,Xtr_sc,balanced_meta=True)
def stackprob(Xa): return S['meta'].predict_proba(np.column_stack([S['xg'].predict_proba(Xa)[:,1],S['lg'].predict_proba(Xa)[:,1],S['rf'].predict_proba(Xa)[:,1],S['mf'].predict(sc.transform(Xa),verbose=0).ravel()]))[:,1]
# 2020 aligned with MISMATCHED encoding (LabelEncoder), 2015 medians for missing features
d20=pd.read_csv(D20); y20=(d20['HeartDisease']=='Yes').astype(int).values
X20r=d20.drop(columns=['HeartDisease']).copy()
for col in X20r.select_dtypes(include='object').columns: X20r[col]=LabelEncoder().fit_transform(X20r[col].astype(str))
FMAP={'BMI':'BMI','Smoking':'Smoker','Stroke':'Stroke','PhysicalHealth':'PhysHlth','MentalHealth':'MentHlth','DiffWalking':'DiffWalk','Sex':'Sex','AgeCategory':'Age','Diabetic':'Diabetes','PhysicalActivity':'PhysActivity','GenHealth':'GenHlth'}
med=d15[FEATS].median(); X20a=pd.DataFrame(index=X20r.index)
for f in FEATS:
    src=next((c20 for c20,c15 in FMAP.items() if c15==f), None)
    X20a[f]=X20r[src].values if (src and src in X20r.columns) else med[f]
X20a=X20a[FEATS]
raw_te=stackprob(Xte_a); raw_20=stackprob(X20a.values)
Xcal,_,ycal,_=train_test_split(Xtr,ytr_a,test_size=0.8,random_state=SEED,stratify=ytr_a)
pL=LogisticRegression(max_iter=1000).fit(stackprob(Xcal.values).reshape(-1,1), ycal.values if hasattr(ycal,'values') else ycal)
pH=LogisticRegression(max_iter=1000).fit(S['meta'].predict_proba(S['oof'])[:,1].reshape(-1,1), ytr_a)
NAIVE={}
for nm,P in [('leaky',pL),('honest',pH)]:
    NAIVE[nm]=dict(base_2015=sens(yte_a,P.predict_proba(raw_te.reshape(-1,1))[:,1]), trans_2020=sens(y20,P.predict_proba(raw_20.reshape(-1,1))[:,1]))
NAIVE['AUC_2020_transport']=round(roc_auc_score(y20,raw_20),3)
print('NAIVE:', NAIVE)

In [ ]:
# ================= HARMONISED TRANSPORT (11-feature, semantic alignment) =================
print('HARMONISED transport: training 11-feature stack...')
SH=['BMI','Smoker','Stroke','PhysHlth','MentHlth','DiffWalk','Sex','Age','Diabetes','PhysActivity','GenHlth']
h15=pd.read_csv(D15)
X15h=pd.DataFrame({'BMI':h15.BMI.astype(float),'Smoker':h15.Smoker.astype(int),'Stroke':h15.Stroke.astype(int),'PhysHlth':h15.PhysHlth.astype(float),'MentHlth':h15.MentHlth.astype(float),'DiffWalk':h15.DiffWalk.astype(int),'Sex':h15.Sex.astype(int),'Age':h15.Age.astype(int),'Diabetes':(h15.Diabetes.astype(float)>0).astype(int),'PhysActivity':h15.PhysActivity.astype(int),'GenHlth':h15.GenHlth.astype(int)})[SH]
y15h=h15.HeartDiseaseorAttack.astype(int).values
h20=pd.read_csv(D20); y20h=(h20.HeartDisease=='Yes').astype(int).values
YN={'No':0,'Yes':1}; GH={'Excellent':1,'Very good':2,'Good':3,'Fair':4,'Poor':5}
AGE={s:i+1 for i,s in enumerate(['18-24','25-29','30-34','35-39','40-44','45-49','50-54','55-59','60-64','65-69','70-74','75-79','80 or older'])}
X20h=pd.DataFrame({'BMI':h20.BMI.astype(float),'Smoker':h20.Smoking.map(YN).astype(int),'Stroke':h20.Stroke.map(YN).astype(int),'PhysHlth':h20.PhysicalHealth.astype(float),'MentHlth':h20.MentalHealth.astype(float),'DiffWalk':h20.DiffWalking.map(YN).astype(int),'Sex':h20.Sex.map({'Female':0,'Male':1}).astype(int),'Age':h20.AgeCategory.map(AGE).astype(int),'Diabetes':h20.Diabetic.map(lambda v:1 if str(v).startswith('Yes') else 0).astype(int),'PhysActivity':h20.PhysicalActivity.map(YN).astype(int),'GenHlth':h20.GenHealth.map(GH).astype(int)})[SH]
Xtr,Xte,ytr,yte=train_test_split(X15h.values,y15h,test_size=0.2,random_state=SEED,stratify=y15h)
sc=StandardScaler().fit(Xtr); Xtr_sc=sc.transform(Xtr)
S=train_stack(Xtr,ytr,Xtr_sc,balanced_meta=False)   # NB06 meta has NO class_weight
def sp(Xa): return S['meta'].predict_proba(np.column_stack([S['xg'].predict_proba(Xa)[:,1],S['lg'].predict_proba(Xa)[:,1],S['rf'].predict_proba(Xa)[:,1],S['mf'].predict(sc.transform(Xa),verbose=0).ravel()]))[:,1]
raw_te=sp(Xte); raw_20=sp(X20h.values)
Xcal,_,ycal,_=train_test_split(X15h.values,y15h,test_size=0.8,random_state=SEED,stratify=y15h)
pL=LogisticRegression(max_iter=1000).fit(sp(Xcal).reshape(-1,1), ycal)
pH=LogisticRegression(max_iter=1000).fit(S['meta'].predict_proba(S['oof'])[:,1].reshape(-1,1), ytr)
HARM={}
for nm,P in [('leaky',pL),('honest',pH)]:
    HARM[nm]=dict(base_2015=sens(yte,P.predict_proba(raw_te.reshape(-1,1))[:,1]), trans_2020=sens(y20h,P.predict_proba(raw_20.reshape(-1,1))[:,1]))
HARM['AUC_2015']=round(roc_auc_score(yte,raw_te),3); HARM['AUC_2020_transport']=round(roc_auc_score(y20h,raw_20),3)
print('HARMONISED:', HARM)

In [ ]:
# ================= SUMMARY =================
for nm,R in [('NAIVE (21-feat)',NAIVE),('HARMONISED (11-feat)',HARM)]:
    print(f'\n{nm}:')
    print(f'  leaky  : 2015 {R["leaky"]["base_2015"]} -> 2020 {R["leaky"]["trans_2020"]}  (drop {round(R["leaky"]["base_2015"]-R["leaky"]["trans_2020"],3)})')
    print(f'  honest : 2015 {R["honest"]["base_2015"]} -> 2020 {R["honest"]["trans_2020"]}  (drop {round(R["honest"]["base_2015"]-R["honest"]["trans_2020"],3)})')
out={'naive':NAIVE,'harmonised':HARM,'threshold':PT,'calibration':'out-of-fold cross-fitting'}
os.makedirs(f'{REPO}/results/brfss2020/temporal_transport/tables',exist_ok=True)
json.dump(out,open(f'{REPO}/results/brfss2020/temporal_transport/tables/transport_leakagefree.json','w'),indent=2)
print('\nSaved transport_leakagefree.json')